In [1]:
import numpy as np
import cv2 as cv
import sys

In [2]:
sys.path.append("/home/alexis/Documents/COMP558/COMP558-FinalProject/code/")
# sys.path.append("/workspaces/python-opencv/repo/code/")

In [3]:
from optical_flow.bounding_box import BoundingBox
from optical_flow.bounding_box import drotrack_bbox_init
from optical_flow.bounding_box import drotrack_bbox_step
from optical_flow.bounding_box import center_to_bbox

In [4]:
video_name = "/home/alexis/Documents/master/vision/repo/libfreenect/wrappers/python/out/VIDEO-20250113-190349.mp4"
cap = cv.VideoCapture(video_name)

In [5]:
# params for ShiTomasi corner detection
feature_params = dict( maxCorners = 1000,
                       qualityLevel = 0.01,
                       minDistance = 5,
                       blockSize = 5 )

# feature_params = {"maxCorners": 10000, "qualityLevel": 0.001, "minDistance": 5}

# Parameters for lucas kanade optical flow
lk_params = dict( winSize  = (15, 15),
                  maxLevel = 2,
                  criteria = (cv.TERM_CRITERIA_EPS | cv.TERM_CRITERIA_COUNT, 10, 0.03))

lk_params = {}

# Create some random colors
color = np.random.randint(0, 255, (100, 3))

# Take first frame and find corners in it
ret, old_frame = cap.read()
old_gray = cv.cvtColor(old_frame, cv.COLOR_BGR2GRAY)
p0 = cv.goodFeaturesToTrack(old_gray, mask = None, **feature_params)

In [6]:
from feature_detection.shi_tomasi import ShiTomasiDetector

detector = ShiTomasiDetector(params = feature_params)
detected = detector.detect_features(old_frame)

print(np.all(np.squeeze(p0) == detected))

True


In [7]:
print(p0.shape)
p_test = np.squeeze(p0)

(174, 1, 2)


In [8]:
x, y, w, h = 275, 200, 110, 85

subset_x = np.logical_and(p0[:, 0, 0] >= x, p0[:, 0, 0] <= x + w)
subset_y = np.logical_and(p0[:, 0, 1] >= y, p0[:, 0, 1] <= y + h)
subset = np.logical_and(subset_x, subset_y)

p0 = p0[subset, ...]

In [9]:
from optical_flow.bounding_box import subset_points
p_test_sub = subset_points(p_test, BoundingBox(x, y, w, h))
print(np.all(np.squeeze(p0) == p_test_sub))

True


In [10]:
prev_bbox = BoundingBox(x, y, w, h)
stats = drotrack_bbox_init(old_frame, np.squeeze(p0), prev_bbox)

In [11]:
fps = 30

fourcc = cv.VideoWriter_fourcc(*'mp4v')
video_writer = cv.VideoWriter("out/lk_test_box.mp4", fourcc, fps, old_frame.shape[:-1][::-1])

In [12]:
from optical_flow.lucas_kanade import LucasKanade

lk = LucasKanade(lk_params=lk_params)

In [13]:
# Create a mask image for drawing purposes
mask = np.zeros_like(old_frame)

while(1):
    ret, frame = cap.read()
    if not ret:
        print('No frames grabbed!')
        break
    frame_gray = cv.cvtColor(frame, cv.COLOR_BGR2GRAY)
    old_gray = cv.cvtColor(old_frame, cv.COLOR_BGR2GRAY)
    # calculate optical flow
    p1, st, err = cv.calcOpticalFlowPyrLK(old_gray, frame_gray, p0, None, **lk_params)
    _, _, _, test_points = lk.track_frame(old_frame, frame, np.squeeze(p0))

    # Select good points
    if p1 is not None:
        good_new = p1[st==1]
        good_old = p0[st==1]


    if not len(good_new.shape) == len(test_points.shape):
        print("ERROR")
        print(good_new.shape)
        print(test_points.shape)
        break

    if not np.all(good_new == test_points):
        print("Error")
        break

    img2 = frame.copy()
    # draw the tracks
    for i, (new, old) in enumerate(zip(good_new, good_old)):
        a, b = new.ravel()
        c, d = old.ravel()
        mask = cv.line(mask, (int(a), int(b)), (int(c), int(d)), color[i].tolist(), 2)
        img2 = cv.circle(img2, (int(a), int(b)), 5, color[i].tolist(), -1)

    # avg = np.mean(good_new, axis=0)
    # avg = np.int_(avg)
    # x, y = avg.ravel()
    # frame = cv.circle(frame, (x, y), 5, (255, 0, 0), -1)

    # img = cv.add(frame, mask)
    # video_writer.write(img)

    # Now update the previous frame and previous points
    old_gray = frame_gray.copy()
    p0 = good_new.reshape(-1, 1, 2)

    bbox_center, stats = drotrack_bbox_step(frame, prev_bbox, np.squeeze(p0), stats)
    prev_bbox = center_to_bbox(bbox_center[0], bbox_center[1], prev_bbox.w, prev_bbox.h)

    img2 = cv.rectangle(img2, (prev_bbox.x, prev_bbox.y), (prev_bbox.x + prev_bbox.w, prev_bbox.y + prev_bbox.h), 255, 2)
    video_writer.write(img2)
    old_frame = frame.copy()

No frames grabbed!


In [14]:
video_writer.release()